<h1> Finding the coordinates associated with api <h1>

This program is to find the coordinates of oil well in texas. There is no well level production data, so will use a centroid of ccordinates in a well to find an approximate lease level coordinates. 

We have generated api numbers in texas_well_lease_api.ipynb. The out put have lease number + dist no (unique id for lease) and api number for each wells. We can find coordinates of api number using digital map data (shape files from well layers by county) from https://www.rrc.texas.gov/resource-center/research/data-sets-available-for-download/. Then merge the total production data with it to find a lease coordinate centroid.

Well layers by county is a set of 255 seperate files. The output is saved in multiple useful forms in the folder coordinates inside data/raw/texas.

In [2]:
from pathlib import Path
import re
import zipfile
import warnings
 
import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm

Location of of the imput files and location where output files will be stored

In [13]:
production_path = Path("../../../../data/raw/texas/cleaned_data/texas_total_prod.parquet")
lease_well_path = Path("../../../../data/raw/texas/cleaned_data/well_api_lease.parquet")
shapefile_zip_folder = Path("../../../../data/raw/texas/Wells")

output_folder = Path("../../../../data/raw/texas/cleaned_data")
output_folder.mkdir(parents=True, exist_ok=True)

In [14]:
prod = pd.read_parquet(production_path)
lease_wells = pd.read_parquet(lease_well_path)

print("Production shape:", prod.shape)
print("Lease/well shape:", lease_wells.shape)

print("\nProduction columns:")
print(prod.columns.tolist())

print("\nLease/well columns:")
print(lease_wells.columns.tolist())

Production shape: (19086019, 10)
Lease/well shape: (587614, 9)

Production columns:
['oil_gas_code', 'district_no', 'lease_no', 'field_no', 'lease_oil_prod_vol', 'lease_csgd_prod_vol', 'lease_csgd_tot_disp', 'operator_no', 'operator_name', 'date']

Lease/well columns:
['OIL_GAS_CODE', 'DISTRICT_NO', 'LEASE_NO', 'WELL_NO', 'API_COUNTY_CODE', 'API_UNIQUE_NO', 'COUNTY_NAME', 'WELLBORE_LOCATION_CODE', 'API_NO']


In [15]:
prod.columns = prod.columns.str.lower()
lease_wells.columns = lease_wells.columns.str.lower()

Production file is lease-level and lease/well file links leases to API numbers. We need a stable lease key to identify them between files. We do oil/gas code + district code + lease code.

In [16]:
def clean_text_series(s):
    return (
        s.astype("string")
         .str.strip()
         .str.upper()
         .str.replace(r"\.0$", "", regex=True)
    )

def normalize_district(s):
    s = clean_text_series(s)
    
    # RRC districts can be 01, 02, 03, 04, 05, 06, 7B, 7C, 08, 8A, 09, 10, etc.
    # If purely numeric, pad to 2 digits. Leave 8A, 7B, etc. as-is.
    return s.apply(lambda x: x.zfill(2) if pd.notna(x) and x.isdigit() else x)

def normalize_lease_no(s):
    s = clean_text_series(s)
    
    # Many RRC lease numbers are 5 digits.
    # If your source uses a different convention, inspect before changing.
    return s.apply(lambda x: x.zfill(5) if pd.notna(x) and x.isdigit() else x)

def normalize_oil_gas_code(s):
    return clean_text_series(s)

In [17]:
for df in [prod, lease_wells]:
    df["oil_gas_code_norm"] = normalize_oil_gas_code(df["oil_gas_code"])
    df["district_no_norm"] = normalize_district(df["district_no"])
    df["lease_no_norm"] = normalize_lease_no(df["lease_no"])

    df["lease_key"] = (
        df["oil_gas_code_norm"] + "_" +
        df["district_no_norm"] + "_" +
        df["lease_no_norm"]
    )

In [18]:
prod[["oil_gas_code", "district_no", "lease_no", "lease_key"]].head()

,oil_gas_code,district_no,lease_no,lease_key
0,O,03,24870,O_03_24870
1,O,03,24870,O_03_24870
2,O,03,24870,O_03_24870
3,O,04,01097,O_04_01097
4,O,04,01097,O_04_01097


In [19]:
lease_wells[["oil_gas_code", "district_no", "lease_no", "lease_key"]].head()


,oil_gas_code,district_no,lease_no,lease_key
0,O,08,00277,O_08_00277
1,O,08,00277,O_08_00277
2,O,08,00287,O_08_00287
3,O,08,00291,O_08_00291
4,O,08,00291,O_08_00291


The texas RRC shapefile have an 8 digit API number (usually it is 10 digit, but texas strips the state code 42 from the API keys). The 8 digit API code is 3 digits for county and 5 for lease. Note that each lease is unique in a district and a district can have many counties. So we need an API key that uniquely identify with a lease_key

In [20]:
def clean_digits(value): 
    """
    Clean a scalar value and return only digits as a string.
    """
    if pd.isna(value):
        return pd.NA
    
    text = str(value).strip()
    text = re.sub(r"\.0$", "", text)
    text = re.sub(r"\D", "", text)
    
    if text == "":
        return pd.NA
    
    return text


def clean_digits_series(s):
    return (
        s.astype("string")
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
         .str.replace(r"\D", "", regex=True)
    )


def normalize_rrc_api8(value):
    """
    Normalize an API-like value to RRC GIS API format:
    
        API8 = county FIPS/API county code 3 digits + unique API number 5 digits
    
    Handles:
    - 8-digit RRC API: CCCNNNNN
    - 10-digit Texas API with state code: 42CCCNNNNN
    - 10-digit RRC API10: CCCNNNNNSS, where SS is sidetrack code
    - 12-digit APINUM: 42CCCNNNNNSS
    """
    digits = clean_digits(value)
    
    if pd.isna(digits):
        return pd.NA
    
    # RRC GIS API field: county3 + unique5
    if len(digits) == 8:
        return digits
    
    # Could be either:
    # - 42 + API8
    # - API8 + sidetrack2
    if len(digits) == 10:
        if digits.startswith("42"):
            return digits[2:10]
        else:
            return digits[:8]
    
    # RRC APINUM: 42 + API8 + sidetrack2
    if len(digits) >= 12 and digits.startswith("42"):
        return digits[2:10]
    
    return pd.NA

In [21]:
county = clean_digits_series(lease_wells["api_county_code"]).str.zfill(3)
unique = clean_digits_series(lease_wells["api_unique_no"]).str.zfill(5)

lease_wells["api8_from_components"] = county + unique

# Invalid rows become <NA>
lease_wells.loc[
    lease_wells["api8_from_components"].isna() |
    (lease_wells["api8_from_components"].str.len() != 8),
    "api8_from_components"
] = pd.NA


In [23]:
lease_wells["api8_from_api_no"] = lease_wells["api_no"].apply(normalize_rrc_api8)
lease_wells["api8"] = lease_wells["api8_from_components"]

lease_wells.loc[
    lease_wells["api8"].isna(),
    "api8"
] = lease_wells["api8_from_api_no"]

In [24]:
lease_wells[
    [
        "api_county_code",
        "api_unique_no",
        "api_no",
        "api8_from_components",
        "api8_from_api_no",
        "api8"
    ]
].head(20)

,api_county_code,api_unique_no,api_no,api8_from_components,api8_from_api_no,api8
0,049,31721,04931721,04931721,04931721,04931721
1,049,32950,04932950,04932950,04932950,04932950
2,049,05559,04905559,04905559,04905559,04905559
3,049,05919,04905919,04905919,04905919,04905919
4,049,80560,04980560,04980560,04980560,04980560
5,049,30486,04930486,04930486,04930486,04930486
6,049,04704,04904704,04904704,04904704,04904704
7,049,35384,04935384,04935384,04935384,04935384
8,049,05369,04905369,04905369,04905369,04905369
9,049,80563,04980563,04980563,04980563,04980563


In [25]:
valid_api8_rate = lease_wells["api8"].notna().mean()
print(f"Lease/well rows with valid API8: {valid_api8_rate:.2%}")

Lease/well rows with valid API8: 100.00%


The coordinate information is in 255 shape files. The are saved as zipped files in data/raw/texas/Wells. The following is to read these zip files.

In [26]:
zip_files = sorted(shapefile_zip_folder.glob("*.zip"))

print(f"Found {len(zip_files)} zip files")
print(zip_files[:5])

Found 255 zip files
[PosixPath('../../../../data/raw/texas/Wells/well001.zip'), PosixPath('../../../../data/raw/texas/Wells/well003.zip'), PosixPath('../../../../data/raw/texas/Wells/well005.zip'), PosixPath('../../../../data/raw/texas/Wells/well007.zip'), PosixPath('../../../../data/raw/texas/Wells/well009.zip')]


Geopandas track the coordinates in a column 'geometry' in lower case. If we make it upper case, it will have conflict with what is default. We need to upper case everything else to normalise it with the rest of the files.

In [27]:
def uppercase_non_geometry_columns(gdf):
    """
    Uppercase all non-geometry columns while preserving the active geometry column.
    """
    geom_col = gdf.geometry.name
    
    rename_map = {
        col: col.upper()
        for col in gdf.columns
        if col != geom_col
    }
    
    gdf = gdf.rename(columns=rename_map)
    
    # Make sure the active geometry column is still set
    gdf = gdf.set_geometry(geom_col)
    
    return gdf

In [28]:
gdfs = []
errors = []

for zip_path in tqdm(zip_files):
    try:
        with zipfile.ZipFile(zip_path) as z:
            shp_names = [
                name for name in z.namelist()
                if name.lower().endswith(".shp")
            ]
        
        if len(shp_names) == 0:
            errors.append((zip_path.name, "No .shp file found inside zip"))
            continue
        
        for shp_name in shp_names:
            zip_uri = f"zip://{zip_path.as_posix()}!{shp_name}"
            
            try:
                gdf = gpd.read_file(zip_uri, engine="pyogrio")
            except Exception:
                gdf = gpd.read_file(zip_uri)
            
            if gdf.empty:
                continue
            
            # Preserve geometry column correctly
            gdf = uppercase_non_geometry_columns(gdf)
            
            # Add source metadata
            gdf["SOURCE_ZIP"] = zip_path.name
            gdf["SOURCE_SHP"] = shp_name
            
            # RRC manual says:
            # Projection: Geographic
            # Units: Decimal Degrees
            # Datum: NAD27
            #
            # If CRS is missing, assign NAD27.
            if gdf.crs is None:
                gdf = gdf.set_crs("EPSG:4267")
            
            # Convert to WGS84 lon/lat
            gdf = gdf.to_crs("EPSG:4326")
            
            gdfs.append(gdf)
            
    except Exception as e:
        errors.append((zip_path.name, str(e)))

print(f"Read {len(gdfs)} shapefile layers")
print(f"Errors: {len(errors)}")
errors[:10]

  0%|          | 0/255 [00:00<?, ?it/s]

Read 733 shapefile layers
Errors: 0


[]

In [30]:
if len(gdfs) == 0:
    raise RuntimeError(
        "No shapefile layers were read. Check the errors list, folder path, and zip contents."
    )

wells_geo_all = pd.concat(gdfs, ignore_index=True)

# Convert back to GeoDataFrame, preserving whatever the geometry column is called
wells_geo_all = gpd.GeoDataFrame(
    wells_geo_all,
    geometry=gdfs[0].geometry.name,
    crs="EPSG:4326"
)

print("Combined shape:", wells_geo_all.shape)
print("CRS:", wells_geo_all.crs)
print("Geometry column:", wells_geo_all.geometry.name)

print("\nGeometry types:")
print(wells_geo_all.geometry.geom_type.value_counts(dropna=False))

print("\nColumns:")
print(wells_geo_all.columns.tolist())

Combined shape: (2958487, 20)
CRS: EPSG:4326
Geometry column: geometry

Geometry types:
Point              2772561
LineString          185748
MultiLineString        178
Name: count, dtype: int64

Columns:
['BOTTOM_ID', 'SURFACE_ID', 'SYMNUM', 'APINUM', 'RELIAB', 'API10', 'API', 'LONG27', 'LAT27', 'LONG83', 'LAT83', 'OUT_FIPS', 'CWELLNUM', 'RADIOACT', 'WELLID', 'STCODE', 'geometry', 'SOURCE_ZIP', 'SOURCE_SHP', 'SHAPE_LEN']


The well zip files contain surface well data as point geometry, bottom well data as point geometry and surface to bottom in more complex geometry. We only need to keep point geometry.

In [31]:
wells_geo = wells_geo_all[
    wells_geo_all.geometry.notna() &
    wells_geo_all.geometry.geom_type.isin(["Point", "MultiPoint"])
].copy()

print("Point well records:", wells_geo.shape)
print(wells_geo.geometry.geom_type.value_counts())

Point well records: (2772561, 20)
Point    2772561
Name: count, dtype: int64


In [32]:
wells_geo["geometry"] = wells_geo.geometry.apply(
    lambda geom: geom.geoms[0] if geom.geom_type == "MultiPoint" else geom
)

wells_geo = gpd.GeoDataFrame(wells_geo, geometry="geometry", crs="EPSG:4326")

The shape files contain 3 types of API <br>
API      length 8  <br>
API10    length 10 <br>
APINUM   length 12 <br>

We need to construct an API key that match with the previous ones.

In [33]:
def coalesce_api8_from_row(row):
    """
    Prefer API, then API10, then APINUM, then other possible API-ish names.
    Return normalized RRC API8.
    """
    candidate_cols = [
        "API",
        "API10",
        "APINUM",
        "API_NO",
        "APINO",
        "API_NUM",
        "API_NUMBER",
        "UWI"
    ]
    
    for col in candidate_cols:
        if col in row.index:
            value = normalize_rrc_api8(row[col])
            if pd.notna(value):
                return value
    
    return pd.NA

In [34]:
wells_geo["api8"] = wells_geo.apply(coalesce_api8_from_row, axis=1)

print("Well GIS rows with API8:", wells_geo["api8"].notna().mean())
wells_geo[["api8"] + [c for c in ["API", "API10", "APINUM", "SOURCE_ZIP", "SOURCE_SHP"] if c in wells_geo.columns]].head()

Well GIS rows with API8: 0.7380663581432474


,api8,API,API10,APINUM,SOURCE_ZIP,SOURCE_SHP
0,00132761,00132761,00132761,4200132761,well001.zip,well001b.shp
1,00101708,00101708,00101708,4200101708,well001.zip,well001b.shp
2,00100126,00100126,00100126,4200100126,well001.zip,well001b.shp
3,00131523,00131523,00131523,4200131523,well001.zip,well001b.shp
4,00132335,00132335,00132335,4200132335,well001.zip,well001b.shp


In [35]:
wells_geo = wells_geo.dropna(subset=["api8"]).copy()

In [36]:
wells_geo["longitude"] = wells_geo.geometry.x
wells_geo["latitude"] = wells_geo.geometry.y

In [37]:
wells_geo[["longitude", "latitude"]].describe()

,longitude,latitude
count,2.046334e+06,2.046334e+06
mean,-9.937510e+01,3.163559e+01
std,2.519270e+00,2.067646e+00
min,-1.065368e+02,2.585656e+01
25%,-1.015927e+02,3.035809e+01
50%,-9.920437e+01,3.200190e+01
75%,-9.764186e+01,3.294880e+01
max,-9.352909e+01,3.649904e+01


Making sure that there is no wells beyond the texas boundary limits

In [38]:
suspicious_coords = wells_geo[
    ~wells_geo["longitude"].between(-107, -93) |
    ~wells_geo["latitude"].between(25, 37)
]

print("Suspicious coordinate rows:", len(suspicious_coords))
suspicious_coords[["api8", "longitude", "latitude", "SOURCE_ZIP", "SOURCE_SHP"]].head()


Suspicious coordinate rows: 0


,api8,longitude,latitude,SOURCE_ZIP,SOURCE_SHP


Isolating surface well from bottom wells. The Shape files contain information about surface or bottom. We need to identify what the marker and create a label for it.

In [39]:
wells_geo["source_shp_lower"] = wells_geo["SOURCE_SHP"].astype(str).str.lower()

wells_geo["well_layer_type"] = "unknown"

wells_geo.loc[
    wells_geo["source_shp_lower"].str.contains(r"s\.shp$", regex=True),
    "well_layer_type"
] = "surface"

wells_geo.loc[
    wells_geo["source_shp_lower"].str.contains(r"b\.shp$", regex=True),
    "well_layer_type"
] = "bottom"

wells_geo["well_layer_type"].value_counts(dropna=False)

well_layer_type
bottom     1031641
surface    1014693
Name: count, dtype: int64

In [40]:
surface_wells_geo = wells_geo[wells_geo["well_layer_type"].isin(["surface", "unknown"])].copy()

print("Surface/unknown well records:", surface_wells_geo.shape)

Surface/unknown well records: (1014693, 25)


In [41]:
wells_geo["SOURCE_SHP"].drop_duplicates().head(50)

0         well001b.shp
5571      well001s.shp
10993     well003b.shp
42017     well003s.shp
69633     well005b.shp
70484     well005s.shp
71149     well007b.shp
73020     well007s.shp
74669     well009b.shp
114850    well009s.shp
155016    well011b.shp
155062    well011s.shp
155101    well013b.shp
166129    well013s.shp
175097    well015b.shp
177149    well015s.shp
178980    well017b.shp
179027    well017s.shp
179075    well019b.shp
179135    well019s.shp
179191    well021b.shp
181805    well021s.shp
184321    well023b.shp
189432    well023s.shp
194520    well025b.shp
200958    well025s.shp
207079    well027b.shp
207169    well027s.shp
207259    well029b.shp
219512    well029s.shp
231756    well031b.shp
231767    well031s.shp
231782    well033b.shp
236679    well033s.shp
240684    well035b.shp
240821    well035s.shp
240919    well037b.shp
241376    well037s.shp
241806    well039b.shp
250810    well039s.shp
258945    well041b.shp
264214    well041s.shp
266667    well043b.shp
266771    w

There can be duplicates.For example there could be a well which appear in both surface and bottom layers of the shape file. We want to prefer surface layer records.

In [42]:
surface_wells_geo["layer_priority"] = surface_wells_geo["well_layer_type"].map({
    "surface": 1,
    "unknown": 2,
    "bottom": 3
}).fillna(9)

sort_cols = ["api8", "layer_priority"]

if "RELIAB" in surface_wells_geo.columns:
    sort_cols.append("RELIAB")

well_coords_one = (
    surface_wells_geo
    .sort_values(sort_cols)
    .drop_duplicates(subset=["api8"], keep="first")
    .copy()
)

In [43]:
keep_cols = [
    "api8",
    "longitude",
    "latitude",
    "geometry",
    "SOURCE_ZIP",
    "SOURCE_SHP",
    "well_layer_type"
]

for optional_col in ["API", "API10", "APINUM", "LAT27", "LONG27", "LAT83", "LONG83", "RELIAB", "SYMBOL", "SYMNUN", "SYMNUM", "WELLID"]:
    if optional_col in well_coords_one.columns and optional_col not in keep_cols:
        keep_cols.append(optional_col)

well_coords_one = well_coords_one[keep_cols].copy()
well_coords_one = gpd.GeoDataFrame(well_coords_one, geometry="geometry", crs="EPSG:4326")

print("Unique API8 coordinate records:", well_coords_one.shape)
well_coords_one.head()

Unique API8 coordinate records: (1014636, 17)


,api8,longitude,latitude,geometry,SOURCE_ZIP,SOURCE_SHP,well_layer_type,API,API10,APINUM,LAT27,LONG27,LAT83,LONG83,RELIAB,SYMNUM,WELLID
5664,00100001,-96.032325,32.001892,POINT (-96.03232 32.00189),well001.zip,well001s.shp,surface,00100001,NaN,NaN,32.001734,-96.032076,32.001894,-96.032320,30,4.0,00001
6033,00100008,-96.009801,31.979878,POINT (-96.0098 31.97988),well001.zip,well001s.shp,surface,00100008,NaN,NaN,31.979719,-96.009553,31.979880,-96.009797,15,10.0,00008
8422,00100037,-95.548114,31.557517,POINT (-95.54811 31.55752),well001.zip,well001s.shp,surface,00100037,NaN,NaN,31.557344,-95.547880,31.557520,-95.548110,15,7.0,00037
9699,00100038,-95.994286,32.008876,POINT (-95.99429 32.00888),well001.zip,well001s.shp,surface,00100038,NaN,NaN,32.008718,-95.994038,32.008878,-95.994281,15,7.0,00038
9698,00100039,-95.991410,32.008862,POINT (-95.99141 32.00886),well001.zip,well001s.shp,surface,00100039,NaN,NaN,32.008704,-95.991162,32.008864,-95.991405,15,7.0,00039


In [44]:
lease_wells_geo = lease_wells.merge(
    well_coords_one,
    on="api8",
    how="left",
    suffixes=("", "_gis")
)

lease_wells_geo = gpd.GeoDataFrame(
    lease_wells_geo,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Lease/well rows:", lease_wells.shape)
print("Lease/well rows with coordinates:", lease_wells_geo["geometry"].notna().sum())
print(f"Coordinate match rate: {lease_wells_geo['geometry'].notna().mean():.2%}")

Lease/well rows: (587614, 16)
Lease/well rows with coordinates: 584135
Coordinate match rate: 99.41%


In [45]:
unmatched_lease_wells = lease_wells_geo[lease_wells_geo["geometry"].isna()].copy()

unmatched_lease_wells[
    [
        "api_county_code",
        "api_unique_no",
        "api_no",
        "api8",
        "county_name",
        "oil_gas_code",
        "district_no",
        "lease_no"
    ]
].head(30)

unmatched_lease_wells[
    [
        "api_county_code",
        "api_unique_no",
        "api_no",
        "api8",
        "county_name",
        "oil_gas_code",
        "district_no",
        "lease_no"
    ]
].shape

(3479, 8)

In [46]:
lease_wells_geo.drop(columns="geometry").to_parquet(
    output_folder / "lease_well_coordinates.parquet",
    index=False
)

lease_wells_geo.to_parquet(
    output_folder / "lease_well_coordinates.geoparquet"
)

No we can combine it with the productionm data. Notice that if there is 5 wells in a lease this will create five rows for that lease for that month. But that does not mean that each well in the lease produced the full lease volume. So the output of this part should be handled with care..

In [47]:
prod_well_points = prod.merge(
    lease_wells_geo[
        [
            "lease_key",
            "api8",
            "well_no",
            "county_name",
            "longitude",
            "latitude",
            "geometry"
        ]
    ],
    on="lease_key",
    how="left"
)

prod_well_points = gpd.GeoDataFrame(
    prod_well_points,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Original production rows:", len(prod))
print("Production rows after well join:", len(prod_well_points))
print(f"Production rows with well coordinates: {prod_well_points['geometry'].notna().mean():.2%}")


Original production rows: 19086019
Production rows after well join: 128016621
Production rows with well coordinates: 99.35%


In order to deal with the over counting due to multiple wells in a lease, we will take the centroid of all the well coordinates in a lease to get a lease level coordinate information.

In [48]:
lease_points = lease_wells_geo.dropna(subset=["geometry"]).copy()
lease_points = gpd.GeoDataFrame(lease_points, geometry="geometry", crs="EPSG:4326")

print("Lease/well point rows with geometry:", lease_points.shape)

Lease/well point rows with geometry: (584135, 32)


In [49]:
lease_wells_geo = lease_wells_geo.set_geometry("geometry")

In [50]:
if lease_wells_geo.crs is None:
    lease_wells_geo = lease_wells_geo.set_crs("EPSG:4326")

In [51]:
lease_points = lease_wells_geo[
    lease_wells_geo["geometry"].notna()
].copy()

lease_points = gpd.GeoDataFrame(
    lease_points,
    geometry="geometry",
    crs=lease_wells_geo.crs
)

print("Rows with geometry:", len(lease_points))
print("Unique leases with geometry:", lease_points["lease_key"].nunique())

Rows with geometry: 584135
Unique leases with geometry: 169051


In [52]:
required_cols = ["lease_key", "api8", "county_name"]

for col in required_cols:
    if col not in lease_points.columns:
        print(f"Missing column: {col}")

In [53]:
group_cols = {
    "n_wells_with_coordinates": ("api8", "nunique")
}

if "county_name" in lease_points.columns:
    group_cols["county_name"] = ("county_name", "first")

if "oil_gas_code_norm" in lease_points.columns:
    group_cols["oil_gas_code_norm"] = ("oil_gas_code_norm", "first")

if "district_no_norm" in lease_points.columns:
    group_cols["district_no_norm"] = ("district_no_norm", "first")

if "lease_no_norm" in lease_points.columns:
    group_cols["lease_no_norm"] = ("lease_no_norm", "first")

Counting wells per lease

In [54]:
lease_well_counts = (
    lease_points
    .groupby("lease_key")
    .agg(**group_cols)
    .reset_index()
)

lease_well_counts.head()

,lease_key,n_wells_with_coordinates,county_name,oil_gas_code_norm,district_no_norm,lease_no_norm
0,O_01_00002,4,WILSON,O,01,00002
1,O_01_00003,2,WILSON,O,01,00003
2,O_01_00005,2,WILSON,O,01,00005
3,O_01_00006,1,WILSON,O,01,00006
4,O_01_00013,1,ATASCOSA,O,01,00013


In [55]:
# Project to Texas Centric Albers before centroid calculation
lease_points_proj = lease_points.to_crs("EPSG:3083")

# Union all well points per lease
lease_geom_proj = (
    lease_points_proj
    .groupby("lease_key")["geometry"]
    .apply(lambda x: x.unary_union.centroid)
    .reset_index()
)

lease_geom_proj = gpd.GeoDataFrame(
    lease_geom_proj,
    geometry="geometry",
    crs="EPSG:3083"
)

# Convert centroid back to lon/lat
lease_geom = lease_geom_proj.to_crs("EPSG:4326")

/var/folders/kz/7z7d62sj31zcvz9s8c3mm0hh0000gn/T/ipykernel_29578/3957310485.py:8: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .apply(lambda x: x.unary_union.centroid)


In [56]:
lease_centroids = lease_well_counts.merge(
    lease_geom,
    on="lease_key",
    how="left"
)

lease_centroids = gpd.GeoDataFrame(
    lease_centroids,
    geometry="geometry",
    crs="EPSG:4326"
)

lease_centroids["lease_longitude"] = lease_centroids.geometry.x
lease_centroids["lease_latitude"] = lease_centroids.geometry.y

lease_centroids.head()

,lease_key,n_wells_with_coordinates,county_name,oil_gas_code_norm,district_no_norm,lease_no_norm,geometry,lease_longitude,lease_latitude
0,O_01_00002,4,WILSON,O,01,00002,POINT (-97.89215 29.2689),-97.892153,29.268898
1,O_01_00003,2,WILSON,O,01,00003,POINT (-97.88844 29.26798),-97.888436,29.267982
2,O_01_00005,2,WILSON,O,01,00005,POINT (-97.89387 29.26598),-97.893871,29.265983
3,O_01_00006,1,WILSON,O,01,00006,POINT (-97.89482 29.26636),-97.894825,29.266355
4,O_01_00013,1,ATASCOSA,O,01,00013,POINT (-98.70805 29.088),-98.708055,29.088001


Creating a small file which count number of wells per lease labelled by lease key with a centroid coordinates for lease location

In [60]:
no_of_wells_per_lease = lease_centroids[['lease_key','n_wells_with_coordinates','lease_latitude','lease_longitude']].copy()

no_of_wells_per_lease.to_parquet(
    output_folder / "wells_per_lease.parquet",
    index=False
)

Making a file combining this with production data without halting structure

In [61]:
prod_lease_centroids = prod.merge(
    lease_centroids[
        [
            "lease_key",
            "n_wells_with_coordinates",
            "county_name",
            "lease_longitude",
            "lease_latitude",
            "geometry"
        ]
    ],
    on="lease_key",
    how="left"
)

prod_lease_centroids = gpd.GeoDataFrame(
    prod_lease_centroids,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Production rows:", len(prod_lease_centroids))
print(f"Production rows with lease centroid: {prod_lease_centroids['geometry'].notna().mean():.2%}")

prod_lease_centroids.head()

Production rows: 19086019
Production rows with lease centroid: 99.17%


,oil_gas_code,district_no,lease_no,field_no,lease_oil_prod_vol,lease_csgd_prod_vol,lease_csgd_tot_disp,operator_no,operator_name,date,oil_gas_code_norm,district_no_norm,lease_no_norm,lease_key,n_wells_with_coordinates,county_name,lease_longitude,lease_latitude,geometry
0,O,03,24870,32688470,115,NaN,0,386310,HILCORP ENERGY COMPANY,2026-01-01,O,03,24870,O_03_24870,1.0,GALVESTON,-95.074364,29.363433,POINT (-95.07436 29.36343)
1,O,03,24870,32688470,349,NaN,0,386310,HILCORP ENERGY COMPANY,2026-02-01,O,03,24870,O_03_24870,1.0,GALVESTON,-95.074364,29.363433,POINT (-95.07436 29.36343)
2,O,03,24870,32688470,203,NaN,0,386310,HILCORP ENERGY COMPANY,2026-03-01,O,03,24870,O_03_24870,1.0,GALVESTON,-95.074364,29.363433,POINT (-95.07436 29.36343)
3,O,04,01097,36108001,278,NaN,0,386310,HILCORP ENERGY COMPANY,2026-01-01,O,04,01097,O_04_01097,7.0,DUVAL,-98.636975,27.944445,POINT (-98.63698 27.94445)
4,O,04,01097,36108001,391,NaN,0,386310,HILCORP ENERGY COMPANY,2026-02-01,O,04,01097,O_04_01097,7.0,DUVAL,-98.636975,27.944445,POINT (-98.63698 27.94445)


In [62]:
prod_lease_centroids.drop(columns="geometry").to_parquet(
    output_folder / "tot_prod_with_lease_coord.parquet",
    index=False
)

prod_lease_centroids.to_parquet(
    output_folder / "tot_prod_with_lease_coord.geoparquet"
)

In [63]:
prod_leases = prod[
    ["lease_key", "oil_gas_code", "district_no", "lease_no"]
].drop_duplicates()

lease_coord_check = prod_leases.merge(
    lease_centroids[["lease_key"]].assign(has_coordinates=True),
    on="lease_key",
    how="left"
)

# Important: force real boolean dtype
lease_coord_check["has_coordinates"] = (
    lease_coord_check["has_coordinates"]
    .fillna(False)
    .astype(bool)
)

print(f"Production leases with coordinates: {lease_coord_check['has_coordinates'].mean():.2%}")

Production leases with coordinates: 99.55%


In [64]:
missing_prod_leases = lease_coord_check.loc[
    ~lease_coord_check["has_coordinates"]
].copy()

missing_prod_leases.shape

(593, 5)

In [66]:
prod_lease_centroids.dropna(subset=["geometry"]).sample(
    min(5000, prod_lease_centroids["geometry"].notna().sum()),
    random_state=42
).explore(
    tiles="CartoDB positron",
    tooltip=[
        "lease_key",
        "county_name",
        "lease_oil_prod_vol",
        "n_wells_with_coordinates"
    ]
)